In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D4 — Eurostat LFS Metadata Workbook
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import pandas as pd

In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D4"

DOCUMENT_NAME = (
    "Eurostat — LFS Metadata Excel — lfsa_esms"
)

SOURCE_SHEET = "Metadata"

EXPECTED_SHEETS = [
    "Metadata",
    "Parameters",
    "Annexes"
]

EXPECTED_REFERENCE_RECORD_COUNT = 83

EXPECTED_HEADER_RECORD_COUNT = 8

EXPECTED_CONCEPT_RECORD_COUNT = 75

REFERENCE_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted",
    "Source Location"
]

ALLOWED_PUBLICATION_FLAGS = {
    "YES",
    "NO"
}

STRUCTURAL_LABELS = {
    "Header",
    "Concepts",
    "Concept name"
}

OUTPUT_DIR = Path(
    "outputs_D4_stage1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Source sheet:", SOURCE_SHEET)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print("Output directory:", OUTPUT_DIR)

Document: D4
Source sheet: Metadata
Expected reference records: 83
Output directory: outputs_D4_stage1


In [3]:
# ============================================================
# 2. Upload source document
# ============================================================

print(
    "Upload the original D4 XLSX workbook."
)

uploaded = files.upload()

xlsx_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".xlsx")
]

if len(xlsx_files) != 1:

    raise ValueError(
        "Upload exactly one XLSX workbook."
    )

SOURCE_PATH = xlsx_files[0]

print(
    "Loaded workbook:",
    SOURCE_PATH.name
)

Upload the original D4 XLSX workbook.


Saving D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx to D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx
Loaded workbook: D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx


In [4]:
# ============================================================
# 3. Source-file hash
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    hash_object = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            hash_object.update(
                chunk
            )

    return hash_object.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print(
    "Workbook SHA-256:",
    SOURCE_SHA256
)

Workbook SHA-256: 040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9


In [5]:
# ============================================================
# 4. Load workbook structure
# ============================================================

excel_file = pd.ExcelFile(
    SOURCE_PATH
)

sheet_names = (
    excel_file.sheet_names
)

expected_sheets_present = all(
    sheet in sheet_names
    for sheet in EXPECTED_SHEETS
)

print(
    "Workbook sheets:",
    sheet_names
)

print(
    "Expected sheets present:",
    expected_sheets_present
)


if not expected_sheets_present:

    raise ValueError(
        "The workbook does not contain all expected sheets."
    )

Workbook sheets: ['Metadata', 'Parameters', 'Annexes']
Expected sheets present: True


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
# ============================================================
# 5. Document metadata
# ============================================================

DOCUMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "file_name":
        SOURCE_PATH.name,

    "file_format":
        "XLSX",

    "source_file_sha256":
        SOURCE_SHA256,

    "number_of_sheets":
        len(
            sheet_names
        ),

    "sheet_names":
        sheet_names,

    "source_sheet_for_reference_values":
        SOURCE_SHEET,

    "text_extractable":
        True,

    "ocr_required":
        False,
}


print(
    json.dumps(
        DOCUMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "document_name": "Eurostat — LFS Metadata Excel — lfsa_esms",
  "file_name": "D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx",
  "file_format": "XLSX",
  "source_file_sha256": "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9",
  "number_of_sheets": 3,
  "sheet_names": [
    "Metadata",
    "Parameters",
    "Annexes"
  ],
  "source_sheet_for_reference_values": "Metadata",
  "text_extractable": true,
  "ocr_required": false
}


In [7]:
# ============================================================
# 6. Sheet-level characterisation
# ============================================================

sheet_characterisation = []

for sheet_name in sheet_names:

    sheet_df = pd.read_excel(
        SOURCE_PATH,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
        keep_default_na=False
    )

    total_cells = int(
        sheet_df.shape[0]
        * sheet_df.shape[1]
    )

    non_empty_mask = sheet_df.apply(
        lambda column:
            column.map(
                lambda value:
                    value is not None
                    and str(value).strip() != ""
            )
    )

    non_empty_cells = int(
        non_empty_mask.sum().sum()
    )

    empty_cells = (
        total_cells
        - non_empty_cells
    )

    text_cells = int(
        sheet_df.apply(
            lambda column:
                column.map(
                    lambda value:
                        isinstance(
                            value,
                            str
                        )
                        and value.strip() != ""
                )
        ).sum().sum()
    )

    numeric_cells = int(
        sheet_df.apply(
            lambda column:
                column.map(
                    lambda value:
                        isinstance(
                            value,
                            (int, float)
                        )
                        and not isinstance(
                            value,
                            bool
                        )
                )
        ).sum().sum()
    )

    sheet_characterisation.append({
        "Sheet Name":
            sheet_name,

        "Rows":
            int(
                sheet_df.shape[0]
            ),

        "Columns":
            int(
                sheet_df.shape[1]
            ),

        "Total Cells":
            total_cells,

        "Non-empty Cells":
            non_empty_cells,

        "Empty Cells":
            empty_cells,

        "Text Cells":
            text_cells,

        "Numeric Cells":
            numeric_cells,

        "Text Density":
            round(
                text_cells
                / non_empty_cells,
                3
            )
            if non_empty_cells
            else 0.0,

        "Numerical Density":
            round(
                numeric_cells
                / non_empty_cells,
                3
            )
            if non_empty_cells
            else 0.0
    })


sheet_characterisation_df = pd.DataFrame(
    sheet_characterisation
)

display(
    sheet_characterisation_df
)

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Sheet Name,Rows,Columns,Total Cells,Non-empty Cells,Empty Cells,Text Cells,Numeric Cells,Text Density,Numerical Density
0,Metadata,86,3,258,231,27,231,0,1.0,0.0
1,Parameters,84,5,420,412,8,412,0,1.0,0.0
2,Annexes,2,4,8,7,1,7,0,1.0,0.0


In [8]:
# ============================================================
# 7. Inspection of raw worksheets
# ============================================================

for sheet_name in sheet_names:

    print(
        "\n"
        + "=" * 80
    )

    print(
        sheet_name
    )

    print(
        "=" * 80
    )

    sheet_df = pd.read_excel(
        SOURCE_PATH,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
        keep_default_na=False
    )

    print(
        "Shape:",
        sheet_df.shape
    )

    display(
        sheet_df.head(
            20
        )
    )


Metadata
Shape: (86, 3)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2
0,Header,,
1,Filename,LFSA_ESMS_A_4D_2019_0000,
2,Published name,lfsa_esms,
3,Data flow ID,LFSA_ESMS_A,
4,Data flow version,1.0,
5,Organization code,4D0,
6,Time dimension,2019-A0,
7,For publication,YES,
8,Data set action,,
9,Concepts,,


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



Parameters
Shape: (84, 5)


,0,1,2,3,4
0,Element name,Element code,Type,Position,Editable
1,Filename,FILENAME,Name,2,
2,Published name,PUBLISH_NAME,String,3,
3,Data flow ID,DATAFLOW_CODE,DataFlow_CODE,4,
4,Data flow version,DATAFLOW_VERSION,DataFlow_VERSION,5,
5,Organization code,DATA_PROVIDER,DataProvider,6,
6,Time dimension,TIME_PERIOD,TimeDimension,7,
7,For publication,FOR_PUBLICATION,YES_NO,8,
8,Data set action,DATA_SET_ACTION,DataSetAction,9,
9,1. Contact,CONTACT,String,12,NO



Annexes
Shape: (2, 4)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3
0,Concept,Annex Type,Annex Name,Annex Description
1,,URL,http://ec.europa.eu/eurostat/statistics-explai...,EU-LFS (Satistics Explained) webpage (addition...


In [9]:
# ============================================================
# 8. Loading Metadata worksheet
# ============================================================

metadata_raw = pd.read_excel(
    SOURCE_PATH,
    sheet_name=SOURCE_SHEET,
    header=None,
    dtype=object,
    keep_default_na=False
)


if metadata_raw.shape[1] < 3:

    raise ValueError(
        "The Metadata worksheet must contain "
        "at least three columns."
    )


metadata_raw = metadata_raw.iloc[
    :,
    :3
].copy()

metadata_raw.columns = [
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]


metadata_raw[
    "Worksheet Row"
] = range(
    1,
    len(metadata_raw) + 1
)


print(
    "Physical Metadata rows:",
    len(metadata_raw)
)

display(
    metadata_raw.head(
        20
    )
)

Physical Metadata rows: 86


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Concept Name,Concept Value,Publication Restricted,Worksheet Row
0,Header,,,1
1,Filename,LFSA_ESMS_A_4D_2019_0000,,2
2,Published name,lfsa_esms,,3
3,Data flow ID,LFSA_ESMS_A,,4
4,Data flow version,1.0,,5
5,Organization code,4D0,,6
6,Time dimension,2019-A0,,7
7,For publication,YES,,8
8,Data set action,,,9
9,Concepts,,,10


In [10]:
# ============================================================
# 9. Defining controlled cell-cleaning utilities
# ============================================================

def preserve_cell_text(value):
    """
    Preserve source-cell content while converting truly
    empty cells to None.

    HTML-like markup and internal line breaks are retained.
    """

    if value is None:

        return None

    if isinstance(value, str):
        if value.strip() == "":
            return None

        # Preserve non-empty source text exactly as stored.
        return value

def normalise_publication_flag(value):
    """
    Preserve YES or NO publication flags.

    Empty source cells remain None.
    """

    cleaned = preserve_cell_text(
        value
    )

    if cleaned is None:

        return None

    upper_value = cleaned.upper()

    if upper_value not in (
        ALLOWED_PUBLICATION_FLAGS
    ):

        return cleaned

    return upper_value

In [11]:
# ============================================================
# 10. Defining section mapping
# ============================================================

TOP_LEVEL_SECTION_PATTERN = re.compile(
    r"^(\d+)\.\s+(.+)$"
)


def build_section_mapping(
    metadata_dataframe
):
    """
    Map each numbered metadata section to its full
    top-level section label.
    """

    section_mapping = {}

    for concept_name in metadata_dataframe[
        "Concept Name"
    ]:

        concept_name = preserve_cell_text(
            concept_name
        )

        if concept_name is None:

            continue

        match = TOP_LEVEL_SECTION_PATTERN.fullmatch(
            concept_name
        )

        if match:

            section_number = match.group(
                1
            )

            section_mapping[
                section_number
            ] = concept_name

    return section_mapping


SECTION_MAPPING = build_section_mapping(
    metadata_raw
)


print(
    json.dumps(
        SECTION_MAPPING,
        indent=2,
        ensure_ascii=False
    )
)

{
  "1": "1. Contact",
  "2": "2. Metadata update",
  "3": "3. Statistical presentation",
  "4": "4. Unit of measure",
  "5": "5. Reference Period",
  "6": "6. Institutional Mandate",
  "7": "7. Confidentiality",
  "8": "8. Release policy",
  "9": "9. Frequency of dissemination",
  "10": "10. Accessibility and clarity",
  "11": "11. Quality management",
  "12": "12. Relevance",
  "13": "13. Accuracy",
  "14": "14. Timeliness and punctuality",
  "15": "15. Coherence and comparability",
  "16": "16. Cost and Burden",
  "17": "17. Data revision",
  "18": "18. Statistical processing",
  "19": "19. Comment"
}


In [12]:
# ============================================================
# 11. Assigning section labels
# ============================================================

def assign_section(
    concept_name
):
    """
    Assign workbook-header fields to Header and numbered
    metadata concepts to their top-level section.
    """

    concept_name = preserve_cell_text(
        concept_name
    )

    if concept_name is None:

        return None

    numbered_match = re.match(
        r"^(\d+)(?:\.\d+)?\.",
        concept_name
    )

    if numbered_match:

        section_number = (
            numbered_match.group(
                1
            )
        )

        return SECTION_MAPPING.get(
            section_number,
            section_number
        )

    return "Header"

In [13]:
# ============================================================
# 12. Reference dataset
# ============================================================

reference_rows = []

for _, source_row in (
    metadata_raw.iterrows()
):

    concept_name = preserve_cell_text(
        source_row[
            "Concept Name"
        ]
    )

    if concept_name is None:

        continue

    if concept_name in STRUCTURAL_LABELS:

        continue

    worksheet_row = int(
        source_row[
            "Worksheet Row"
        ]
    )

    concept_value = preserve_cell_text(
        source_row[
            "Concept Value"
        ]
    )

    publication_restricted = (
        normalise_publication_flag(
            source_row[
                "Publication Restricted"
            ]
        )
    )

    reference_rows.append({
        "Section":
            assign_section(
                concept_name
            ),

        "Concept Name":
            concept_name,

        "Concept Value":
            concept_value,

        "Publication Restricted":
            publication_restricted,

        "Source Location":
            (
                f"{SOURCE_SHEET}!"
                f"A{worksheet_row}:"
                f"C{worksheet_row}"
            )
    })


reference_values_df = pd.DataFrame(
    reference_rows,
    columns=REFERENCE_FIELDS
)


print(
    "Reference records:",
    len(reference_values_df)
)

display(
    reference_values_df
)

Reference records: 83


,Section,Concept Name,Concept Value,Publication Restricted,Source Location
0,Header,Filename,LFSA_ESMS_A_4D_2019_0000,None,Metadata!A2:C2
1,Header,Published name,lfsa_esms,None,Metadata!A3:C3
2,Header,Data flow ID,LFSA_ESMS_A,None,Metadata!A4:C4
3,Header,Data flow version,1.0,None,Metadata!A5:C5
4,Header,Organization code,4D0,None,Metadata!A6:C6
...,...,...,...,...,...
78,18. Statistical processing,18.3. Data collection,"<p style=""text-align: justify;"">Please refer t...",NO,Metadata!A82:C82
79,18. Statistical processing,18.4. Data validation,"<p>Please refer to the ESMS page on <a title=""...",NO,Metadata!A83:C83
80,18. Statistical processing,18.5. Data compilation,"<p style=""margin: 0cm 0cm 3pt;"">For each count...",NO,Metadata!A84:C84
81,18. Statistical processing,18.6. Adjustment,<p>No adjustments are made to the EU-LFS data....,NO,Metadata!A85:C85


In [14]:
# ============================================================
# 13. Count of reference-record categories
# ============================================================

header_record_count = int(
    (
        reference_values_df[
            "Section"
        ]
        == "Header"
    ).sum()
)

concept_record_count = int(
    len(reference_values_df)
    - header_record_count
)


print(
    "Header records:",
    header_record_count
)

print(
    "Concept records:",
    concept_record_count
)

print(
    "Total records:",
    len(reference_values_df)
)

Header records: 8
Concept records: 75
Total records: 83


In [15]:
# ============================================================
# 14. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract every metadata record represented in the Metadata worksheet of
the attached Eurostat LFS metadata workbook.

For each record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Use only the Metadata worksheet.
- Include the workbook-header metadata records in rows 2–9.
- Include every metadata concept record in rows 12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude the structural row labelled Header.
- Exclude the structural row labelled Concepts.
- Exclude the column-heading row containing Concept name, Concept value
  and Restricted from publication.
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.
- Preserve Concept Name exactly as represented in the source worksheet.
- Preserve Concept Value exactly as represented, including HTML-like
  tags, hyperlinks, entities, punctuation and internal line breaks.
- Preserve YES and NO publication-restriction flags.
- Use null when a Concept Value or Publication Restricted cell is
  genuinely empty.
- Do not clean HTML, rewrite labels, infer values, follow hyperlinks,
  calculate values or use external information.
- Return exactly 83 records.
- Return the result as valid JSON using the exact field names defined
  in the extraction schema.
- Return one record for every included Metadata worksheet row.
- Do not include explanations before or after the JSON.
"""

print(
    EXTRACTION_TASK
)


Extract every metadata record represented in the Metadata worksheet of
the attached Eurostat LFS metadata workbook.

For each record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Use only the Metadata worksheet.
- Include the workbook-header metadata records in rows 2–9.
- Include every metadata concept record in rows 12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude the structural row labelled Header.
- Exclude the structural row labelled Concepts.
- Exclude the column-heading row containing Concept name, Concept value
  and Restricted from publication.
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.
- Preserve Concept Name exactly as represented in the source worksheet.
- Preserve Concept Value exactly as represented, including HTML-like
  tags, hyperlinks, entities, punctuation and internal line breaks.
- Preserve YES and NO

In [16]:
# ============================================================
# 15. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "Metadata worksheet row",

    "source_scope":
        (
            "Metadata worksheet rows 2–9 and 12–86"
        ),

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Section":
            (
                "Header or the full top-level numbered "
                "metadata section label."
            ),

        "Concept Name":
            (
                "Metadata concept or workbook-header "
                "field label exactly as represented."
            ),

        "Concept Value":
            (
                "Associated source-cell value. HTML-like "
                "markup is preserved. Empty cells are null."
            ),

        "Publication Restricted":
            (
                "YES, NO, or null when the source cell "
                "is empty."
            ),

        "Source Location":
            (
                "Metadata worksheet source-cell range "
                "supporting the reference record."
            )
    },

    "excluded_content": [
        "Metadata row 1: Header",
        "Metadata row 10: Concepts",
        "Metadata row 11: column headings",
        "Parameters worksheet records",
        "Annexes worksheet records"
    ]
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "record_level": "Metadata worksheet row",
  "source_scope": "Metadata worksheet rows 2–9 and 12–86",
  "expected_record_count": 83,
  "fields": {
    "Section": "Header or the full top-level numbered metadata section label.",
    "Concept Name": "Metadata concept or workbook-header field label exactly as represented.",
    "Concept Value": "Associated source-cell value. HTML-like markup is preserved. Empty cells are null.",
    "Publication Restricted": "YES, NO, or null when the source cell is empty.",
    "Source Location": "Metadata worksheet source-cell range supporting the reference record."
  },
  "excluded_content": [
    "Metadata row 1: Header",
    "Metadata row 10: Concepts",
    "Metadata row 11: column headings",
    "Parameters worksheet records",
    "Annexes worksheet records"
  ]
}


In [17]:
# ============================================================
# 16. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level": "metadata worksheet record",
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Section": {
            "type": ["string", "null"]
        },

        "Concept Name": {
            "type": ["string", "null"]
        },

        "Concept Value": {
            "type": ["string", "null"]
        },

        "Publication Restricted": {
            "type": ["string", "null"],
            "allowed_values": [
                "YES",
                "NO"
            ]
        }
    },

    "expected_output_structure": {
        "document_id": DOCUMENT_ID,

        "records": [
            {
                "Section": "string or null",
                "Concept Name": "string or null",
                "Concept Value": "string or null",
                "Publication Restricted": "YES, NO or null"
            }
        ]
    }
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "record_level": "metadata worksheet record",
  "expected_record_count": 83,
  "fields": {
    "Section": {
      "type": [
        "string",
        "null"
      ]
    },
    "Concept Name": {
      "type": [
        "string",
        "null"
      ]
    },
    "Concept Value": {
      "type": [
        "string",
        "null"
      ]
    },
    "Publication Restricted": {
      "type": [
        "string",
        "null"
      ],
      "allowed_values": [
        "YES",
        "NO"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D4",
    "records": [
      {
        "Section": "string or null",
        "Concept Name": "string or null",
        "Concept Value": "string or null",
        "Publication Restricted": "YES, NO or null"
      }
    ]
  }
}


In [18]:
# ============================================================
# 17. Validation of schema and record counts
# ============================================================

actual_fields = (
    reference_values_df.columns.tolist()
)

schema_valid = (
    actual_fields
    == REFERENCE_FIELDS
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

header_count_valid = (
    header_record_count
    == EXPECTED_HEADER_RECORD_COUNT
)

concept_count_valid = (
    concept_record_count
    == EXPECTED_CONCEPT_RECORD_COUNT
)


print(
    "Schema valid:",
    schema_valid
)

print(
    "Record count valid:",
    record_count_valid
)

print(
    "Header count valid:",
    header_count_valid
)

print(
    "Concept count valid:",
    concept_count_valid
)


if not schema_valid:

    raise ValueError(
        "The D4 reference schema is invalid."
    )

if not record_count_valid:

    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} "
        f"records but found "
        f"{len(reference_values_df)}."
    )

if not header_count_valid:

    raise ValueError(
        "Unexpected number of header records."
    )

if not concept_count_valid:

    raise ValueError(
        "Unexpected number of concept records."
    )

Schema valid: True
Record count valid: True
Header count valid: True
Concept count valid: True


In [19]:
# ============================================================
# 18. Validation of values and publication flags
# ============================================================

missing_concept_names = int(
    reference_values_df[
        "Concept Name"
    ].isna().sum()
)

missing_sections = int(
    reference_values_df[
        "Section"
    ].isna().sum()
)

observed_non_null_flags = set(
    reference_values_df[
        "Publication Restricted"
    ].dropna()
)

unexpected_publication_flags = sorted(
    observed_non_null_flags
    - ALLOWED_PUBLICATION_FLAGS
)

duplicate_source_locations = (
    reference_values_df[
        "Source Location"
    ].duplicated(
        keep=False
    )
)


print(
    "Missing concept names:",
    missing_concept_names
)

print(
    "Missing sections:",
    missing_sections
)

print(
    "Unexpected publication flags:",
    unexpected_publication_flags
)

print(
    "Duplicate source locations:",
    int(
        duplicate_source_locations.sum()
    )
)


if missing_concept_names > 0:

    raise ValueError(
        "Reference records contain missing Concept Names."
    )

if missing_sections > 0:

    raise ValueError(
        "Reference records contain missing Sections."
    )

if unexpected_publication_flags:

    raise ValueError(
        "Unexpected publication flags detected: "
        f"{unexpected_publication_flags}"
    )

if duplicate_source_locations.any():

    raise ValueError(
        "Duplicate source locations were detected."
    )

Missing concept names: 0
Missing sections: 0
Unexpected publication flags: []
Duplicate source locations: 0


In [20]:
# ============================================================
# 19. Creation of representation indicators
# ============================================================

HTML_TAG_PATTERN = r"<[^>]+>"

HTML_ENTITY_PATTERN = (
    r"&(?:nbsp|amp|lt|gt|quot|apos);"
)


records_with_html_tags = int(
    reference_values_df[
        "Concept Value"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        HTML_TAG_PATTERN,
        regex=True
    )
    .sum()
)


records_with_html_entities = int(
    reference_values_df[
        "Concept Value"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        HTML_ENTITY_PATTERN,
        regex=True,
        case=False
    )
    .sum()
)


restricted_records = int(
    reference_values_df[
        "Publication Restricted"
    ]
    .eq(
        "YES"
    )
    .sum()
)


non_restricted_records = int(
    reference_values_df[
        "Publication Restricted"
    ]
    .eq(
        "NO"
    )
    .sum()
)


null_restriction_records = int(
    reference_values_df[
        "Publication Restricted"
    ]
    .isna()
    .sum()
)


records_with_missing_concept_value = int(
    reference_values_df[
        "Concept Value"
    ]
    .isna()
    .sum()
)


METADATA_REPRESENTATION_INDICATORS = {
    "document_id":
        DOCUMENT_ID,

    "records_with_html_tags":
        records_with_html_tags,

    "records_with_html_entities":
        records_with_html_entities,

    "restricted_records":
        restricted_records,

    "non_restricted_records":
        non_restricted_records,

    "records_with_null_restriction_flag":
        null_restriction_records,

    "records_with_missing_concept_value":
        records_with_missing_concept_value,

    "html_cleaning_applied":
        False,

    "source_text_rewritten":
        False
}


print(
    json.dumps(
        METADATA_REPRESENTATION_INDICATORS,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "records_with_html_tags": 53,
  "records_with_html_entities": 6,
  "restricted_records": 5,
  "non_restricted_records": 70,
  "records_with_null_restriction_flag": 8,
  "records_with_missing_concept_value": 15,
  "html_cleaning_applied": false,
  "source_text_rewritten": false
}


In [21]:
# ============================================================
# 20. Reference summary
# ============================================================

REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "reference_scope":
        (
            "Metadata worksheet rows 2–9 and 12–86"
        ),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(reference_values_df)
        ),

    "record_count_valid":
        bool(
            record_count_valid
        ),

    "schema_valid":
        bool(
            schema_valid
        ),

    "header_records":
        header_record_count,

    "concept_records":
        concept_record_count,

    "number_of_sections":
        int(
            reference_values_df[
                "Section"
            ].nunique()
        ),

    "records_with_missing_concept_value":
        records_with_missing_concept_value,

    "restricted_records":
        restricted_records,

    "non_restricted_records":
        non_restricted_records,

    "records_with_null_restriction_flag":
        null_restriction_records,

    "records_with_html_tags":
        records_with_html_tags,

    "records_with_html_entities":
        records_with_html_entities,

    "reference_values_branch_independent":
        True,

    "reference_values_reused_across_branches":
        True,

    "html_preserved_in_reference_values":
        True
}


print(
    json.dumps(
        REFERENCE_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "document_name": "Eurostat — LFS Metadata Excel — lfsa_esms",
  "reference_scope": "Metadata worksheet rows 2–9 and 12–86",
  "expected_reference_records": 83,
  "observed_reference_records": 83,
  "record_count_valid": true,
  "schema_valid": true,
  "header_records": 8,
  "concept_records": 75,
  "number_of_sections": 20,
  "records_with_missing_concept_value": 15,
  "restricted_records": 5,
  "non_restricted_records": 70,
  "records_with_null_restriction_flag": 8,
  "records_with_html_tags": 53,
  "records_with_html_entities": 6,
  "reference_values_branch_independent": true,
  "reference_values_reused_across_branches": true,
  "html_preserved_in_reference_values": true
}


In [22]:
# ============================================================
# 21. Document characterisation
# ============================================================

DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "file_format":
        "XLSX",

    "number_of_sheets":
        len(
            sheet_names
        ),

    "sheet_names":
        sheet_names,

    "metadata_sheet_rows":
        int(
            metadata_raw.shape[0]
        ),

    "metadata_sheet_columns":
        int(
            metadata_raw.shape[1] - 1
        ),

    "reference_records":
        int(
            len(reference_values_df)
        ),

    "contains_metadata_fields":
        True,

    "contains_parameters_sheet":
        "Parameters" in sheet_names,

    "contains_annexes_sheet":
        "Annexes" in sheet_names,

    "contains_html_like_content":
        records_with_html_tags > 0,

    "contains_publication_restrictions":
        restricted_records > 0,

    "contains_cross_sheet_schema":
        True,

    "ocr_required":
        False,

    "dominant_content_type":
        "structured statistical metadata text",

    "numerical_content":
        "Low",

    "schema_explicitness":
        "High",

    "reference_source_sheet":
        SOURCE_SHEET
}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "file_format": "XLSX",
  "number_of_sheets": 3,
  "sheet_names": [
    "Metadata",
    "Parameters",
    "Annexes"
  ],
  "metadata_sheet_rows": 86,
  "metadata_sheet_columns": 3,
  "reference_records": 83,
  "contains_metadata_fields": true,
  "contains_parameters_sheet": true,
  "contains_annexes_sheet": true,
  "contains_html_like_content": true,
  "contains_publication_restrictions": true,
  "contains_cross_sheet_schema": true,
  "ocr_required": false,
  "dominant_content_type": "structured statistical metadata text",
  "numerical_content": "Low",
  "schema_explicitness": "High",
  "reference_source_sheet": "Metadata"
}


In [23]:
# ============================================================
# 22. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Low",
        "Evidence Source":
            "Workbook inspection + Metadata worksheet profiling",
        "Justification":
            "The Metadata worksheet follows a clear top-to-bottom "
            "sequence in which metadata fields and numbered concepts "
            "appear in a stable logical order."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Medium",
        "Evidence Source":
            "Workbook structure inspection",
        "Justification":
            "The workbook is tabular, but the Metadata worksheet mixes "
            "header records, structural rows, column headings, top-level "
            "section records, and detailed metadata concepts."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Low",
        "Evidence Source":
            "Metadata hierarchy inspection",
        "Justification":
            "The metadata hierarchy is explicitly encoded through "
            "numbered concepts such as 1., 1.1., and related nested "
            "section labels."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "File-format inspection",
        "Justification":
            "The workbook contains natively machine-readable cells, "
            "so image sharpness does not constrain text recovery."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "File-format inspection",
        "Justification":
            "No scanning noise or visual degradation affects the "
            "machine-readable workbook content."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated workbook inspection",
        "Justification":
            "All relevant content is stored as machine-readable "
            "spreadsheet data and OCR is not required."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Low",
        "Evidence Source":
            "Manual metadata inspection",
        "Justification":
            "Eurostat and ESMS metadata terminology is specialised "
            "but used consistently across the Metadata worksheet."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "Medium",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "Concept Name, Concept Value, and Publication Restricted "
            "map naturally to the extraction schema, but header records "
            "and numbered metadata concepts have slightly different "
            "structural roles and Section must be derived from the "
            "metadata hierarchy."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "Low",
        "Evidence Source":
            "Automated workbook profiling",
        "Justification":
            "The Metadata worksheet is predominantly textual and "
            "contains relatively little numerical information."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All information required by the defined extraction task "
            "is present in the source; empty Concept Value cells are "
            "valid for structural or section-level records rather than "
            "unintended missing information."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Low",
        "Evidence Source":
            "Reference and workbook validation",
        "Justification":
            "The Metadata worksheet follows a consistent metadata "
            "organisation and publication-restriction values conform "
            "to the expected conventions."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "High",
        "Evidence Source":
            "Workbook profiling + representation diagnostics",
        "Justification":
            "The workbook combines three worksheets with different "
            "schemas together with hierarchical metadata, HTML-like "
            "markup, hyperlinks, entities, long-form text, coded "
            "parameters, and annex references."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "Medium",
        "Evidence Source":
            "Metadata and representation inspection",
        "Justification":
            "Although conventional measurement units are not central "
            "to the document, metadata labels, codes, HTML conventions, "
            "section numbering, and naming representations require "
            "moderate standardisation."
    }
]

indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

indicator_assessment_df

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Low,Workbook inspection + Metadata worksheet profi...,The Metadata worksheet follows a clear top-to-...
1,Structural Readiness,Table Structure Integrity,Medium,Workbook structure inspection,"The workbook is tabular, but the Metadata work..."
2,Structural Readiness,Section/Header Hierarchy,Low,Metadata hierarchy inspection,The metadata hierarchy is explicitly encoded t...
3,Visual/OCR Readiness,Sharpness,Low,File-format inspection,The workbook contains natively machine-readabl...
4,Visual/OCR Readiness,Noise / Degradation,Low,File-format inspection,No scanning noise or visual degradation affect...
5,Visual/OCR Readiness,OCR Dependency,Low,Automated workbook inspection,All relevant content is stored as machine-read...
6,Semantic Quality,Terminology Consistency,Low,Manual metadata inspection,Eurostat and ESMS metadata terminology is spec...
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,"Concept Name, Concept Value, and Publication R..."
8,Semantic Quality,Numerical Density,Low,Automated workbook profiling,The Metadata worksheet is predominantly textua...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All information required by the defined extrac...


In [24]:
# ============================================================
# 23. Validation of indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: "
        f"{invalid_scores}"
    )


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: "
        f"{missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: "
        f"{unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [25]:
# ============================================================
# 24. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

dimension_assessment_df

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.00,2,Low
1,Representation and Normalisation Complexity,2.50,2,High
2,Semantic Quality,1.33,3,Low
3,Structural Readiness,1.33,3,Low
4,Visual/OCR Readiness,1.00,3,Low


In [26]:
# ============================================================
# 25. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [27]:
# ============================================================
# 26. Reference-construction metadata
# ============================================================

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_sheet":
        SOURCE_SHEET,

    "reference_file":
        "D4_reference_values.csv",

    "reference_construction_method":
        (
            "Deterministic source-cell transcription "
            "followed by manual verification"
        ),

    "reference_scope":
        (
            "Metadata worksheet rows 2–9 and 12–86"
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "html_preserved":
        True,

    "html_cleaning_applied":
        False,

    "publication_flags_preserved":
        True,

    "empty_source_cells_stored_as_null":
        True,

    "parameters_sheet_used_as_reference_records":
        False,

    "annexes_sheet_used_as_reference_records":
        False,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "notes": (
        "The D4 reference dataset is a deterministic "
        "transcription of the included Metadata worksheet "
        "rows. Source HTML, hyperlinks, entities, line breaks, "
        "empty values and publication flags are preserved. "
        "The Parameters and Annexes worksheets are used only "
        "for workbook characterisation."
    )
}


print(
    json.dumps(
        REFERENCE_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D4",
  "document_name": "Eurostat — LFS Metadata Excel — lfsa_esms",
  "source_file": "D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx",
  "source_file_sha256": "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9",
  "source_sheet": "Metadata",
  "reference_file": "D4_reference_values.csv",
  "reference_construction_method": "Deterministic source-cell transcription followed by manual verification",
  "reference_scope": "Metadata worksheet rows 2–9 and 12–86",
  "expected_reference_record_count": 83,
  "observed_reference_record_count": 83,
  "reference_fields": [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted",
    "Source Location"
  ],
  "html_preserved": true,
  "html_cleaning_applied": false,
  "publication_flags_preserved": true,
  "empty_source_cells_stored_as_null": true,
  "parameters_sheet_used_as_reference_records": false,
  "annexes_sheet_used_as_reference_records": false,
  "manual_calculation_applie

In [28]:
# ============================================================
# 27. Run Final integrity checks
# ============================================================

source_locations_valid = bool(
    reference_values_df[
        "Source Location"
    ]
    .str.match(
        r"^Metadata!A\d+:C\d+$"
    )
    .all()
)


reference_integrity_passed = all([
    schema_valid,
    record_count_valid,
    header_count_valid,
    concept_count_valid,
    missing_concept_names == 0,
    missing_sections == 0,
    not unexpected_publication_flags,
    not duplicate_source_locations.any(),
    source_locations_valid
])


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "reference_integrity_passed":
        bool(
            reference_integrity_passed
        ),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_records":
        int(
            len(reference_values_df)
        ),

    "schema_valid":
        bool(
            schema_valid
        ),

    "record_count_valid":
        bool(
            record_count_valid
        ),

    "header_count_valid":
        bool(
            header_count_valid
        ),

    "concept_count_valid":
        bool(
            concept_count_valid
        ),

    "missing_concept_name_count":
        missing_concept_names,

    "missing_section_count":
        missing_sections,

    "unexpected_publication_flag_count":
        len(
            unexpected_publication_flags
        ),

    "duplicate_source_location_count":
        int(
            duplicate_source_locations.sum()
        ),

    "source_locations_valid":
        source_locations_valid
}


print(
    json.dumps(
        REFERENCE_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


if not reference_integrity_passed:

    raise ValueError(
        "The D4 reference dataset failed "
        "the integrity checks."
    )

{
  "document_id": "D4",
  "reference_integrity_passed": true,
  "expected_reference_records": 83,
  "observed_reference_records": 83,
  "schema_valid": true,
  "record_count_valid": true,
  "header_count_valid": true,
  "concept_count_valid": true,
  "missing_concept_name_count": 0,
  "missing_section_count": 0,
  "unexpected_publication_flag_count": 0,
  "duplicate_source_location_count": 0,
  "source_locations_valid": true
}


In [29]:
# ============================================================
# 28. Export D4 reference outputs
# ============================================================

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR
    / "D4_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR
    / "D4_reference_values.json"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D4_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D4_dimension_assessment.csv"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D4_extraction_schema.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR
    / "D4_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D4_reference_summary.json"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR
    / "D4_reference_metadata.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D4_reference_integrity.json"
)

DOCUMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D4_document_metadata.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D4_document_characterisation.json"
)

SHEET_CHARACTERISATION_PATH = (
    OUTPUT_DIR
    / "D4_sheet_characterisation.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR
    / "D4_quality_evidence.json"
)

REPRESENTATION_INDICATORS_PATH = (
    OUTPUT_DIR
    / "D4_metadata_representation_indicators.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR
    / "D4_extraction_task.txt"
)


reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False
)

reference_values_df.to_json(
    REFERENCE_VALUES_JSON_PATH,
    orient="records",
    indent=2,
    force_ascii=False
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False
)

sheet_characterisation_df.to_csv(
    SHEET_CHARACTERISATION_PATH,
    index=False
)


json_outputs = [
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),
    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),
    (
        DOCUMENT_METADATA_PATH,
        DOCUMENT_METADATA
    ),
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    ),
    (
        REPRESENTATION_INDICATORS_PATH,
        METADATA_REPRESENTATION_INDICATORS
    )
]


for output_path, output_content in (
    json_outputs
):

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            output_content,
            file,
            indent=2,
            ensure_ascii=False
        )


with open(
    EXTRACTION_TASK_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        EXTRACTION_TASK.strip()
    )


print(
    "D4 Stage 1 outputs exported."
)

D4 Stage 1 outputs exported.


In [30]:
# ============================================================
# 29. List generated outputs
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    SHEET_CHARACTERISATION_PATH,
    QUALITY_EVIDENCE_PATH,
    REPRESENTATION_INDICATORS_PATH,
    EXTRACTION_TASK_PATH
]


print(
    "Generated D4 Stage 1 files:\n"
)

for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name
    )

Generated D4 Stage 1 files:

- D4_reference_values.csv
- D4_reference_values.json
- D4_indicator_assessment.csv
- D4_dimension_assessment.csv
- D4_extraction_schema.json
- D4_reference_schema.json
- D4_reference_summary.json
- D4_reference_metadata.json
- D4_reference_integrity.json
- D4_document_metadata.json
- D4_document_characterisation.json
- D4_sheet_characterisation.csv
- D4_quality_evidence.json
- D4_metadata_representation_indicators.json
- D4_extraction_task.txt


In [31]:
# ============================================================
# 30. Download outputs
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            output_path
        )

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>